links:
- https://www.geeksforgeeks.org/python/python-loop-through-folders-and-files-in-directory/

notes:
- to handle class imbalance (750/250) tell the loss function to care more about the minority class.
-

## Import packages
---


In [34]:
import pandas as pd
import os
import cv2
import numpy as np
import tensorflow as tf
from keras.src.layers import Rescaling
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from sklearn.utils.class_weight import compute_class_weight

### Load real images
---


In [35]:
dir = "data/real_dataset/"

valid_extensions = (".jpg", ".jpeg", ".png")

# collect all sub-folder paths
sub_folder_paths = []

for root, dirs, files in os.walk(dir):
    for filename in files:
        if filename.lower().endswith(valid_extensions):
            sub_folder_paths.append(os.path.join(root, filename)) # append full path

images_real = [] # numpy Array (from cv2)
for path in sub_folder_paths:
    img = cv2.imread(path)

    if img is not None:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        images_real.append(img)

print(f'loaded {len(images_real)} images')


loaded 745 images


### Load Ai generated images
---


In [36]:
dir = "data/ai_generated_dataset/"

valid_extensions = (".jpg", ".jpeg", ".png")

# collect all sub-folder paths
sub_folder_paths = []

for root, dirs, files in os.walk(dir):
    for filename in files:
        if filename.lower().endswith(valid_extensions):
            sub_folder_paths.append(os.path.join(root, filename)) # append full path

images_ai = []
for path in sub_folder_paths:
    img = cv2.imread(path)

    if img is not None:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        images_ai.append(img)

print(f'loaded {len(images_ai)} images')

loaded 250 images


### Data Augmentation
---
TODO: Mirroring, Noise, Blur, Rotation

# Standardize Image Sizes
----


In [37]:
IMG_SIZE = (224,224)

processed_real = [cv2.resize(img, IMG_SIZE) for img in images_real]
processed_ai = [cv2.resize(img, IMG_SIZE) for img in images_ai]

X_real = np.array(processed_real)
X_ai = np.array(processed_ai)

y_real = np.zeros(len(X_real), dtype=np.int32)  # class 0
y_ai = np.ones(len(X_ai), dtype=np.int32)       # class 1

X_combined = np.concatenate([X_real, X_ai], axis=0)
y_combined = np.concatenate([y_real, y_ai], axis=0)

# Build Tensorflow dataset
---


In [38]:
ds = tf.data.Dataset.from_tensor_slices((X_combined,y_combined))

BUFFER_SIZE = len(X_combined)
SPLIT = 0.8
BATCH_SIZE = 32

ds = ds.shuffle(buffer_size=BUFFER_SIZE)

train_ds = ds.take(int(SPLIT * BUFFER_SIZE))
val_ds = ds.skip(int(SPLIT * BUFFER_SIZE))

train_ds = train_ds.batch(BATCH_SIZE).prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds = val_ds.batch(BATCH_SIZE).prefetch(buffer_size=tf.data.AUTOTUNE)


# Build Tensorflow CNN
---


In [39]:
model = Sequential([
    # Rescale input
    Rescaling(1./255, input_shape=(224,224,3)),
    
    # HiddenLayers 1
    Conv2D(32, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(32, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(2,2),
    Dropout(0.25),
    
    # HiddenLayers 2
    Conv2D(64, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(64, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(2,2),
    Dropout(0.25),
    
    # HiddenLayers 3
    Conv2D(128, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(128, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(2,2),
    Dropout(0.25),
    
    # HiddenLayers 4 - deeper feature extraction
    Conv2D(256, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(256, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(2,2),
    Dropout(0.4),
    
    # Fully connected layers
    Flatten(),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),

    Dense(1, activation='sigmoid') # 0 = real, 1 = ai generated
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

C:\Users\Tobia\Documents\FH Technikum repos\SS2026\CV&NLP\CV_Project\.venv\Lib\site-packages\keras\src\layers\preprocessing\data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


# Train the Model
---


In [40]:
# Compute class weights to handle class imbalance
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_combined),
    y=y_combined
)

class_weight_dict = dict(enumerate(class_weights))

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    class_weight=class_weight_dict
)

Epoch 1/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 77s 3s/step - accuracy: 0.6621 - loss: 0.6896 - val_accuracy: 0.7588 - val_loss: 0.6570
Epoch 2/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 70s 3s/step - accuracy: 0.7864 - loss: 0.4335 - val_accuracy: 0.7286 - val_loss: 1.6833
Epoch 3/10
12/25 ━━━━━━━━━━━━━━━━━━━━ 39s 3s/step - accuracy: 0.8415 - loss: 0.3945

KeyboardInterrupt: 

# Test Data with val_ds
---

In [ ]:
import numpy as np

y_true = []
for _, labels in val_ds:
    y_true.extend(labels.numpy())
y_true = np.array(y_true)

y_pred_probs = model.predict(val_ds)

# 3. Convert probabilities to binary choices (0 or 1)
y_pred = (y_pred_probs >= 0.5).astype(int).flatten()

from sklearn.metrics import classification_report

# Print precision, recall, and F1-score
print(classification_report(y_true, y_pred, target_names=['Real (0)', 'AI (1)']))

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Compute the confusion matrix
cm = confusion_matrix(y_true, y_pred)

# Plot the confusion matrix visually
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Real', 'AI'])
disp.plot(cmap=plt.cm.Blues)
plt.title("Validation Confusion Matrix")
plt.show()

# Save model to disk
---


In [ ]:
# save model to disk
model.save("my_model.keras")

# Load model from disk
---


In [ ]:
model = tf.keras.models.load_model("my_model.keras")

# Visualize some pictures of each class with its classification
---

In [ ]:
# Visualize sample predictions from each class
import matplotlib.pyplot as plt
import random

# Get samples from validation set
real_samples = []
ai_samples = []

for images, labels in val_ds:
    for img, label in zip(images.numpy(), labels.numpy()):
        if label == 0 and len(real_samples) < 4:
            real_samples.append(img)
        elif label == 1 and len(ai_samples) < 4:
            ai_samples.append(img)
        if len(real_samples) == 4 and len(ai_samples) == 4:
            break
    if len(real_samples) == 4 and len(ai_samples) == 4:
        break

# Combine samples
all_samples = real_samples + ai_samples
all_labels = [0]*len(real_samples) + [1]*len(ai_samples)

# Get predictions
predictions = model.predict(np.array(all_samples))
predicted_classes = (predictions >= 0.5).astype(int).flatten()

# Plot
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for idx, (img, true_label, pred_class, pred_prob) in enumerate(zip(all_samples, all_labels, predicted_classes, predictions)):
    axes[idx].imshow(img.astype('uint8'))
    
    true_class_name = "Real" if true_label == 0 else "AI"
    pred_class_name = "Real" if pred_class == 0 else "AI"
    
    # Color: green if correct, red if wrong
    color = 'green' if true_label == pred_class else 'red'
    
    title = f"True: {true_class_name}\nPredicted: {pred_class_name} ({pred_prob[0]:.2%})"
    axes[idx].set_title(title, color=color, fontweight='bold')
    axes[idx].axis('off')

plt.suptitle('Sample Predictions: Real vs AI Images', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()